# Evaluation Retriever System

## 1. Ручная проверка — spot check

In [1]:
from src.pipeline.embedder import Embedder
from src.pipeline.reranker import Reranker
from src.pipeline.sparse_encoder import SparseEncoder
from src.pipeline.chunker import Chunker
from src.qdrant.qdrant import QdrantDB
import os

embedder = Embedder()
reranker = Reranker()
sparse_encoder = SparseEncoder()
db = QdrantDB()

# Fit BM25 на тех же чанках, что в базе
chunker = Chunker(chunk_size=500, chunk_overlap=100)
all_chunks = []
for filename in os.listdir("data/cleaned"):
    with open(f"data/cleaned/{filename}", "r", encoding="utf-8") as f:
        text = f.read()
    source = os.path.splitext(filename)[0]
    all_chunks.extend(chunker.chunk(text, source=source))

sparse_encoder.fit([c.text for c in all_chunks])
print(f"BM25 fitted on {len(all_chunks)} chunks")


def rrf_merge(dense_results: list[dict], bm25_results: list[tuple[int, float]],
              k: int = 60, limit: int = 20) -> list[dict]:
    """Reciprocal Rank Fusion: объединяет dense и BM25 результаты."""
    scores: dict[str, float] = {}
    doc_map: dict[str, dict] = {}

    for rank, r in enumerate(dense_results):
        key = r["text"][:100]
        scores[key] = scores.get(key, 0) + 1.0 / (k + rank + 1)
        doc_map[key] = r

    for rank, (idx, _bm25_score) in enumerate(bm25_results):
        chunk = all_chunks[idx]
        key = chunk.text[:100]
        scores[key] = scores.get(key, 0) + 1.0 / (k + rank + 1)
        if key not in doc_map:
            doc_map[key] = {"text": chunk.text, "source": chunk.source, "score": 0.0}

    sorted_keys = sorted(scores, key=lambda x: scores[x], reverse=True)[:limit]
    results = []
    for key in sorted_keys:
        r = doc_map[key]
        r["score"] = scores[key]
        results.append(r)
    return results


def search(query: str, limit: int = 5, retrieve: int = 20, mode: str = "hybrid"):
    if mode == "hybrid":
        dense_vec = embedder.embed_query(query)
        dense_results = db.search(dense_vec, limit=retrieve)
        bm25_results = sparse_encoder.search(query, limit=retrieve)
        results = rrf_merge(dense_results, bm25_results, limit=retrieve)
    else:
        dense_vec = embedder.embed_query(query)
        results = db.search(dense_vec, limit=retrieve)

    reranked = reranker.rerank(query, results, top_k=limit)
    for i, r in enumerate(reranked):
        print(f"\n--- Result {i+1} (score: {r['score']:.4f}, rerank: {r['rerank_score']:.4f}) [{r['source']}] ---")
        print(r["text"][:300])

W0404 23:11:20.217000 51744 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


BM25 fitted on 7770 chunks


In [2]:
search("Как Фродо получил Кольцо?")


--- Result 1 (score: 0.0297, rerank: 0.9826) [LordOfTheRings] ---
Фродо чувствовал себя идиотом. Не зная, как спасти положение, он ползком пробрался под столами в темный угол к Долгоброду. Тот сидел неподвижно и бесстрастно, ничем не выдавая свои мысли. Фродо прислонился спиной к стене и снял Кольцо. Как оно попало к нему на палец, он понятия не имел. Он лишь пред

--- Result 2 (score: 0.0143, rerank: 0.9656) [LordOfTheRings] ---
— Да, оно у меня, — ответил Фродо так неохотно, что сам себе удивился. — Оно совсем не изменилось.
— Я бы только одним глазком, — повторил Бильбо.
Когда Фродо одевался, он обнаружил Кольцо у себя на груди на легкой, но очень прочной новой цепочке. Теперь он медленно его вытащил. Бильбо протянул руку

--- Result 3 (score: 0.0164, rerank: 0.9518) [LordOfTheRings] ---
Фродо вдруг почувствовал себя в глупейшем положении и обнаружил, что перебирает мелочи в кармане (как обычно, когда говорил Речи). Нащупав Кольцо на цепочке, он вдруг ощутил желание надеть его и та

In [3]:
search("Кто такой Гэндальф?")


--- Result 1 (score: 0.0292, rerank: 0.9199) [LordOfTheRings] ---
— Это кто такой? — спросил Фродо. — Я о нем ни разу не слышал.
— Ты вряд ли мог о нем слышать, — ответил Гэндальв. — Хоббитами он до сих пор не интересовался. Среди Мудрых он один из великих: глава всех магов и глава Совета. Вместе с ученостью возросла его гордыня, он ни с кем не делится знанием и н

--- Result 2 (score: 0.0154, rerank: 0.6628) [LordOfTheRings] ---
— Он? Кто он? — нетерпеливо спросил Фродо.
— Ах, он! Гэндальв, если вы его знаете. Говорят, он маг, но он мой хороший друг, как бы там ни было. А теперь не знаю, что он мне скажет и сделает, если появится; не удивлюсь, если все мое пиво сквасит или меня в колоду превратит, он несколько вспыльчив. Да

--- Result 3 (score: 0.0143, rerank: 0.3414) [LordOfTheRings] ---
— Гэндальв! — воскликнул Эомер. — Гэндальва Серого в Рубежном Крае знают. Но я должен предостеречь тебя: сейчас это имя не откроет тебе двери к сердцу короля. Гэндальв много раз бывал в нашей стран

In [12]:
search("Битва у Хельмовой Пади")


--- Result 1 (score: 0.0143, rerank: 0.1076) [LordOfTheRings] ---
Битва, доставшаяся им, была долгой и жестокой. Южане были храбрыми и сильными воинами, испытанными в боях, яростными в отчаянии, а банды востокан прошли обучение в Мордоре, и никто не собирался сдаваться. Везде: на поле, под горой, за каждым сгоревшим амбаром и у каждого пригорка собирались враги, г

--- Result 2 (score: 0.0164, rerank: 0.0471) [LordOfTheRings] ---
битва на Вершине — Battle of the Peak
битва на Зеленых полях — Battle of GreenFields
битва на полях Гондора — Battle of the Fields of Gondor
битва на полях Пеленнора — Battle of the Pelennor fields
битва под Деревьями — Battle under the Trees in Mirkwood
битва при Келебранте — Battle of the Field of

--- Result 3 (score: 0.0143, rerank: 0.0265) [LordOfTheRings] ---
— Айя-хой! — кричал Гимли. — За стеной орки! Гейя! Сюда, Леголас! Их тут хватит на нас обоих! Казад ай-мену!
По голосу гнома, взвившемуся над шумом битвы, старый Гамлин следил за событиями с верхуш

In [10]:
search("Что такое Шир?")


--- Result 1 (score: 0.7879, rerank: 0.2859) [LordOfTheRings] ---
Название Хоббитшир (по-английски просто Шир —Shire, перевод старохоббитского Suza — Суза) тоже осовременено, как и многие названия местностей в стране хоббитов. Это особых трудностей не представило, потому что названия обычно составляются из различных вполне переводимых элементов. У хоббитов в назва

--- Result 2 (score: 0.7980, rerank: 0.0768) [LordOfTheRings] ---
«О чем ты?» — спросил я.
«Мне сказали, что Всадники везде спрашивают про какой-то Шир».
«Хоббитшир — не какой-то», — сказал я, но сердце у меня упало. Даже Мудрые страшатся встретить Девятерых, когда они собираются в боевой отряд под рукой короля-Чернокнижника. В прошлом колдун и великий вождь, сейч

--- Result 3 (score: 0.7762, rerank: 0.0009) [LordOfTheRings] ---
далее шли:
орэ— сердце, внутренний разум,вала— сила высшего духа,анна— дар,вилья— воздух, небо,ромэн— восток,арда— место, область,ламбэ— язык,альда— дерево,сильмэ— звездный свет,сильмэ нукверна(обр

## 2. Автоматическая оценка на QA-парах

Метрики:
- **Hit Rate@k** — доля вопросов, для которых правильный контекст попал в top-k
- **MRR@k** (Mean Reciprocal Rank) — средняя обратная позиция правильного результата

Сравниваем: только vector search vs vector search + reranker

In [8]:
import json
from tqdm import tqdm

with open("data/qa_pairs.jsonl", "r", encoding="utf-8") as f:
    qa_pairs = [json.loads(line) for line in f]

print(f"Загружено {len(qa_pairs)} QA-пар")

Загружено 292 QA-пар


In [ ]:
from difflib import SequenceMatcher

OVERLAP_THRESHOLD = 0.5


def is_relevant(retrieved_text: str, gold_context: str) -> bool:
    shorter = min(retrieved_text, gold_context, key=len)
    match = SequenceMatcher(None, retrieved_text, gold_context).find_longest_match(
        0, len(retrieved_text), 0, len(gold_context)
    )
    return match.size / len(shorter) > OVERLAP_THRESHOLD


def count_relevant_in_corpus(gold_context: str) -> int:
    """Сколько чанков в корпусе релевантны данному gold контексту."""
    return sum(1 for c in all_chunks if is_relevant(c.text, gold_context))


def evaluate(qa_pairs, top_k=5, retrieve=20, use_reranker=True, mode="dense"):
    hits = {k: 0 for k in [1, 3, 5]}
    mrr_sum = 0.0
    recall_sum = {k: 0.0 for k in [1, 3, 5]}

    for qa in tqdm(qa_pairs):
        query = qa["question"]
        gold_context = qa["context"]

        if mode == "hybrid":
            dense_vec = embedder.embed_query(query)
            dense_results = db.search(dense_vec, limit=retrieve)
            bm25_results = sparse_encoder.search(query, limit=retrieve)
            results = rrf_merge(dense_results, bm25_results, limit=retrieve)
        else:
            vector = embedder.embed_query(query)
            results = db.search(vector, limit=retrieve)

        if use_reranker:
            results = reranker.rerank(query, results, top_k=top_k)
        else:
            results = results[:top_k]

        # Hit Rate & MRR
        first_hit = None
        for i, r in enumerate(results):
            if is_relevant(r["text"], gold_context):
                first_hit = i + 1
                break

        if first_hit:
            mrr_sum += 1.0 / first_hit
            for k in hits:
                if first_hit <= k:
                    hits[k] += 1

        # Recall@k
        total_relevant = count_relevant_in_corpus(gold_context)
        if total_relevant > 0:
            for k in recall_sum:
                found = sum(1 for r in results[:k] if is_relevant(r["text"], gold_context))
                recall_sum[k] += found / total_relevant

    n = len(qa_pairs)
    metrics = {f"Hit Rate@{k}": hits[k] / n for k in hits}
    metrics["MRR@5"] = mrr_sum / n
    # for k in recall_sum:
    #     metrics[f"Recall@{k}"] = recall_sum[k] / n
    # return metrics

In [15]:
# print("=== Dense only ===")
# metrics_dense = evaluate(qa_pairs, use_reranker=False, mode="dense")
# for k, v in metrics_dense.items():
#     print(f"  {k}: {v:.4f}")

# print("\n=== Dense + Reranker ===")
# metrics_dense_rerank = evaluate(qa_pairs, use_reranker=True, mode="dense")
# for k, v in metrics_dense_rerank.items():
#     print(f"  {k}: {v:.4f}")

# print("\n=== Hybrid (Dense + BM25 RRF) ===")
# metrics_hybrid = evaluate(qa_pairs, use_reranker=False, mode="hybrid")
# for k, v in metrics_hybrid.items():
#     print(f"  {k}: {v:.4f}")

print("\n=== Hybrid + Reranker ===")
metrics_hybrid_rerank = evaluate(qa_pairs, use_reranker=True, mode="hybrid")
for k, v in metrics_hybrid_rerank.items():
    print(f"  {k}: {v:.4f}")


=== Hybrid + Reranker ===


100%|██████████| 292/292 [15:06<00:00,  3.10s/it]

  Hit Rate@1: 0.4212
  Hit Rate@3: 0.5171
  Hit Rate@5: 0.5377
  MRR@5: 0.4694
  Recall@1: 0.1665
  Recall@3: 0.2223
  Recall@5: 0.2420


In [11]:
import pandas as pd

df = pd.DataFrame({
    "Метрика": list(metrics_dense.keys()),
    "Dense": [f"{v:.4f}" for v in metrics_dense.values()],
    "Dense + Reranker": [f"{v:.4f}" for v in metrics_dense_rerank.values()],
    "Hybrid (RRF)": [f"{v:.4f}" for v in metrics_hybrid.values()],
    "Hybrid + Reranker": [f"{v:.4f}" for v in metrics_hybrid_rerank.values()],
})
df

,Метрика,Dense,Dense + Reranker,Hybrid (RRF),Hybrid + Reranker
0,Hit Rate@1,0.2705,0.3973,0.2774,0.4212
1,Hit Rate@3,0.4041,0.4795,0.4144,0.5171
2,Hit Rate@5,0.4589,0.4932,0.4760,0.5377
3,MRR@5,0.3394,0.4376,0.3470,0.4694
